# SeeSaw — Notebook 3: GGUF Export

Merges LoRA adapters into `google/gemma-3-1b-it`, converts to GGUF float16, quantises to Q4_K_M, validates, and uploads to GCS.

**Input:** `gs://seesaw-models/checkpoints/seesaw-gemma3-v1/`  
**Output:** `gs://seesaw-models/seesaw-gemma3-1b-q4km.gguf` (~800 MB)

**Runtime:** Free Colab T4, ~20 minutes

See `docs/FINE_TUNING.md` for validation criteria.

In [ ]:
# Step 1: Install dependencies
!pip install -q transformers peft google-cloud-storage huggingface_hub

In [ ]:
# Step 2: Authenticate — GCP + HuggingFace
from google.colab import auth
auth.authenticate_user()

import subprocess
subprocess.run(['gcloud', 'config', 'set', 'project', 'seesaw-3e396'], check=True)

from huggingface_hub import login
login()  # needs read access + gemma-3-1b-it license accepted
print('Auth complete')

In [ ]:
# Step 3: Build llama.cpp (for GGUF conversion + quantisation)
import subprocess

subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/ggerganov/llama.cpp', '/tmp/llama.cpp'], check=True)
subprocess.run(['pip', 'install', '-q', '-r', '/tmp/llama.cpp/requirements.txt'], check=True)
subprocess.run(['cmake', '-B', '/tmp/llama.cpp/build', '/tmp/llama.cpp'], check=True)
subprocess.run(['cmake', '--build', '/tmp/llama.cpp/build', '--config', 'Release', '-j4'], check=True)
print('llama.cpp built successfully')

In [ ]:
# Step 4: Download checkpoint from GCS
import subprocess

GCS_CHECKPOINT = 'gs://seesaw-models/checkpoints/seesaw-gemma3-v1'
LOCAL_CKPT     = '/tmp/seesaw-gemma3-checkpoint'

subprocess.run(['gsutil', '-m', 'cp', '-r', GCS_CHECKPOINT, LOCAL_CKPT], check=True)
print(f'Downloaded to {LOCAL_CKPT}')

In [ ]:
# Step 5: Merge LoRA adapters into base weights
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE_MODEL  = 'google/gemma-3-1b-it'
LOCAL_CKPT  = '/tmp/seesaw-gemma3-checkpoint'
LOCAL_FINAL = '/tmp/seesaw-gemma3-final'

base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float16)
model = PeftModel.from_pretrained(base, LOCAL_CKPT)
merged = model.merge_and_unload()
merged.save_pretrained(LOCAL_FINAL)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.save_pretrained(LOCAL_FINAL)
print(f'Merged model saved to {LOCAL_FINAL}')

In [ ]:
# Step 6: Convert to GGUF float16, then quantise to Q4_K_M
import subprocess, os

LOCAL_FINAL  = '/tmp/seesaw-gemma3-final'
GGUF_F16     = '/tmp/seesaw-gemma3-f16.gguf'
GGUF_Q4KM    = '/tmp/seesaw-gemma3-1b-q4km.gguf'
LLAMA_DIR    = '/tmp/llama.cpp'
QUANTIZE_BIN = f'{LLAMA_DIR}/build/bin/llama-quantize'

# Convert HF model → GGUF float16
subprocess.run(['python', f'{LLAMA_DIR}/convert_hf_to_gguf.py',
                LOCAL_FINAL, '--outfile', GGUF_F16, '--outtype', 'f16'], check=True)

# Quantise to Q4_K_M
subprocess.run([QUANTIZE_BIN, GGUF_F16, GGUF_Q4KM, 'Q4_K_M'], check=True)

size_mb = os.path.getsize(GGUF_Q4KM) / 1024 / 1024
print(f'GGUF Q4_K_M size: {size_mb:.0f} MB (expected ~800 MB)')

In [ ]:
# Step 7: Validate — run sample inference and check JSON output
import subprocess, json, re

LLAMA_CLI = '/tmp/llama.cpp/build/bin/llama-cli'
GGUF_Q4KM = '/tmp/seesaw-gemma3-1b-q4km.gguf'

VALIDATION_PROMPT = (
    '<bos><start_of_turn>user\n'
    'You are SeeSaw, a gentle storytelling companion for children aged 4-8.\n'
    'Child: Vihas, age 5. Objects: teddy_bear, book. Scene: living_room. Continue the story.'
    '<end_of_turn>\n<start_of_turn>model\n'
)

result = subprocess.run(
    [LLAMA_CLI, '-m', GGUF_Q4KM, '-p', VALIDATION_PROMPT, '-n', '200', '--temp', '0.8'],
    capture_output=True, text=True
)
output = result.stdout[-600:]
print('Validation output:', output)

match = re.search(r'\{.*\}', output, re.DOTALL)
if match:
    parsed = json.loads(match.group())
    assert 'story_text' in parsed and 'question' in parsed and 'is_ending' in parsed
    print('✓ JSON valid — story_text, question, is_ending all present')
    word_count = len(parsed['story_text'].split())
    print(f'  story_text word count: {word_count} (target 40-80)')
else:
    print('⚠ JSON not found in output — review model output above')

In [ ]:
# Step 8: Upload to GCS
import subprocess

GGUF_Q4KM = '/tmp/seesaw-gemma3-1b-q4km.gguf'
GCS_DEST  = 'gs://seesaw-models/seesaw-gemma3-1b-q4km.gguf'

subprocess.run(['gsutil', 'cp', GGUF_Q4KM, GCS_DEST], check=True)
print(f'Uploaded to {GCS_DEST}')
print('Next: test GET /model/latest returns the correct signed URL and size.')